# nano Beatrix v1.2 — Pollux dual pretrained with frames in every phase

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_2.ipynb)

Every nano so far met frames in a continuation of 1.0 to 1.5 billion bytes after a 16.1 billion byte plain diet, and a later phase displaced an earlier form (the caption phase moved the old flat-frame copy from 0.81 to 0.47 while its own trie copy rose to 0.96). v1.2 puts the frames into the diet itself: the same body (Pollux dual, from the same birth as v1 and v1.1), the same recipe as v1.1 (Muon on the trunk, Adam on the rest, the output layer's readout on AdamW), the same thirteen phases and bytes, with HALF of every phase's rows framed from the first step in four forms: carry rows (a record planted early and carried through the text, the question at the far end), frames rows (sized, sliced system frames nested in the text), caption windows (the caption pack's trie frames with the frame withheld in half the documents, the question asked) and note rows (a fact of the text framed as a note), in the caption, math and code registers. A framed row takes two of the phase's own rows as its text; the plain rows are the phase's rows verbatim.

| | v1.2 (this notebook) | the control (nano-pollux-dual-v1.1, on the Hub) |
|---|---|---|
| body and birth | Pollux dual, the bed's build at seed 1 | the same |
| recipe | v1.1's | the same |
| phases and bytes | the 2s diet, 16.1 B in 13 phases, 61,422 steps | the same |
| rows | half framed in every phase (`framed_diet.FORMS['p50']`) | all plain |

The reads at every phase boundary (and the bed's gauges every 4,000 steps): the held-out bits beside the control's, the frame copy and nested reads, the far question with the body's own carries, the note read, and the caption read against its prior on the pack's held split. The marks, written before the run: FRAMES KEPT when every frame read ends within 0.10 of its peak over the boundaries (the loss rule), TEXT KEPT when the held-out bits stay within 0.02 of the control's at every boundary and within 0.01 at the end, MEANING when the caption extraction ends 0.20 or more above the prior. C4 prints the table and the marks at any time.

Length: about 7.3 hours on an RTX PRO 6000 at the pace the control ran (610 kB/s), plus the gauges and the caption pack (a few minutes, built once). The run resumes from the Hub when a session ends: rerun C1 to C3.


In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.58"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
from geolip.nano_beatrix import framed_diet as FDT
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 58), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the caption pack reads its sources and the run ships to the repo'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1_2'                            # the session folder: the birth, the pack, the runs, the bed's gauge folder
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the craft under its own name
NAMES = ['nano-pollux-dual-v1.2']                     # half the rows framed; the alternatives: 'nano-pollux-dual-v1.2-p25', 'nano-pollux-dual-v1.2-p75'
FORM = V1.framed_of(NAMES[0])                         # the framed form's key (framed_diet.FORMS)
SHAPES = [(16, 4), (32, 2)]                           # rows a micro-batch x micro-batches a step, each 262,144 bytes; C2 benches them and keeps 16 x 4 unless another is 10% faster
GAUGE_EVERY = V1.GAUGE_EVERY                          # the bed's gauges every 4,000 steps and at every boundary
PACK_ROWS = None                                      # the caption pack: None = the first shards of each source whole (about a gigabyte); a number = that many rows a source (a smoke)
MAX_HOURS = None                                      # this session's budget in hours (None: until the diet ends or the cell is stopped)
PACK = f'{OUT}/caption_pack'
print('decision block:', ', '.join(f"{n} ({V1.ALL[n]['arm']} seed {V1.ALL[n]['seed']}, framed {V1.framed_of(n)})" for n in NAMES),
      '| the readout on AdamW', V1.train_of(NAMES[0]), '| framed', FDT.FORMS[FORM]['p_framed'], 'of the rows, kinds', dict(FDT.FORMS[FORM]['kinds']),
      '| shapes', SHAPES, '| gauges every', GAUGE_EVERY, '| out', OUT, '| ships to', REPO)


In [ ]:
# C2 (P) — PREFLIGHT: the birth (the start of the bed run, checked), the held-out head, THE CAPTION PACK (built once), THE FRAMED STREAM
# on this machine (the fineweb-edu stream wrapped as the run wraps it: its pace against the card's need, the kinds' shares, one decoded
# row of each kind), the 2s step on this card at each shape, the length, the push probe, the card. Nothing trains here.
import json, time
import numpy as np
from huggingface_hub import HfApi, hf_hub_download
from geolip.nano_beatrix.bed_corpus import write_heldout
from geolip.nano_beatrix import caption_pack as CP
from geolip.nano_beatrix import frames_data as FD
from geolip.alephllm.data import special_tokens as ST
from geolip.alephllm.data.streams import build_stream
from geolip.alephllm.data.tokenizers import build_tokenizer
births = {n: V1.write_birth(n, f'{OUT}/births') for n in NAMES}
held = write_heldout(f'{OUT}/bed')
print('births:', ', '.join(f'{n} <- {V1.ALL[n]["arm"]} seed {V1.ALL[n]["seed"]}' for n in NAMES), '| held-out head', os.path.basename(held))
stats = CP.ensure(PACK, HF_TOKEN, PACK_ROWS, log=print)
os.environ[CP.PACK_ENV] = PACK
PK, HELD = CP.load(PACK), CP.load(PACK, 'held')
print(f'the pack: {PK.n:,} training documents; {HELD.n:,} held')

# the framed stream as the run builds it: the fineweb-edu stream at micro-batch 1 under the wrapper
tok = build_tokenizer('byte-trigram')
inner = build_stream('fineweb-edu', tok, V1.CTX, 1, seed=1337)
fs = FDT.FramedStream(inner, 16, 1337, FDT.FORMS[FORM], pack=PK)
NAMES_ST = {v: k for k, v in vars(ST).items() if isinstance(v, int) and 0xF0 <= v <= 0xFF and k.isupper()}
def show(ids):
    return ''.join(f'<{NAMES_ST[c]}>' if c in NAMES_ST else chr(c) if 32 <= c < 127 or c == 10 else f'\\x{c:02x}' for c in ids)
seen, t0, n = {}, time.time(), 0
first = fs.next_batch()                               # the stream's start (the shard's first rows) outside the clock
t0 = time.time()
while time.time() - t0 < 15:
    ids, kind = fs.row()
    n += 1
    if kind not in seen:
        seen[kind] = ids
rate = n / (time.time() - t0)
need = 610_000 / FD.ROW_BYTES                         # rows a second the control's pace needs (610 kB/s on the RTX PRO 6000)
print(f'the framed stream composes {rate:,.0f} rows a second against {need:.0f} needed at the control\'s pace'
      + ('' if rate > 1.5 * need else ' — TOO SLOW: the card would wait on the data'))
print('kinds over', n, 'rows:', FDT.share_table(fs.counts))
for kind, ids in seen.items():
    print(f'\n===== one {kind} row (the first 700 bytes) =====')
    print(show(ids[:700].tolist()))

card_gb = torch.cuda.get_device_properties(0).total_memory / 2 ** 30
BENCH = []
for nm in NAMES:
    for mb, ga in SHAPES:
        r = V1.bench(nm, births[nm], mb, ga)
        BENCH.append(r)
        print(f'  {nm:<26} {mb:>2} x {ga}: {r["tok_s"] / 1e3:6.0f}k bytes a second, {r["step_s"]:.3f} s a step, peak {r["peak_gb"]:.1f} GB')
(MB, GA), why = V1.pick_shape(BENCH, card_gb)
print(f'-> the craft trains at {MB} x {GA}: {why}')
steps = V1.plan_steps(NAMES[0], MB * GA * V1.CTX)
secs = {r['name']: r['step_s'] * steps for r in BENCH if (r['micro_batch'], r['grad_accum']) == (MB, GA)}
print(f'-> the diet trains about {sum(secs.values()) / 3600:.1f} h on this card at the bench pace ({steps:,} steps), plus the gauges, the caption reads and the evals')

# the push probe (the ship-complete law): a write to the repo and a read-back BEFORE any training.
api = HfApi(token=HF_TOKEN)
path = 'preflight/push_probe_v1_2.json'
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0), 'bench': BENCH,
                   'shape': [MB, GA], 'crafts': {n: V1.ALL[n] for n in NAMES}, 'compose_rows_s': rate, 'kinds': fs.counts,
                   'pack_docs': PK.n, 'pack_held': HELD.n}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1.2: push probe')
files = set(api.list_repo_files(REPO))
PUSH_OK = path in files
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)
if PUSH_OK:
    now = open(hf_hub_download(REPO, 'README.md', token=HF_TOKEN), encoding='utf-8').read() if 'README.md' in files else ''
    if 'Castor and Pollux, v1.2' not in now:
        api.upload_file(path_or_fileobj=V1.card_in_training(NAMES).encode(), path_in_repo='README.md', repo_id=REPO,
                        commit_message='nano v1.2: the card while the craft trains')
        print('the card written: README.md (v1.2)')


In [ ]:
# C3 (T) — THE TRAIN: the craft through the thirteen phases, resuming from its record in the repo; the trainer's own lines (the step bar,
# the health readouts, the evals, the checkpoints), the bed's gauge lines and the caption reads ([v1]) print as they come; a BOUNDARY
# returns to the driver, which evals, gauges, reads the captions, ships the boundary table and TensorBoard, and goes on.
# Rerun the cell to continue (a new session too). The cell's stop button saves the running state before it stops.
HIST = V1.run(OUT, HF_TOKEN if PUSH_OK else None, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY,
              max_hours=MAX_HOURS, caption_pack=PACK, caption_pack_rows=PACK_ROWS)
from IPython.display import Markdown, display
CTRL = json.load(open(hf_hub_download(REPO, 'nano-pollux-dual-v1.1/v1_boundaries.json', token=HF_TOKEN), encoding='utf-8'))
for n in NAMES:
    display(Markdown(V1.v12_table(HIST.get(n) or [], CTRL)))
    display(Markdown(V1.v12_marks_table(V1.v12_marks(HIST.get(n) or [], CTRL))))


In [ ]:
# C4 — THE TABLE AND THE MARKS beside the control, and the curves: the boundary rows from the repo (all sessions), the control's rows
# (nano-pollux-dual-v1.1), the marks by the pre-registered rules. Run any time; it trains nothing.
import json
from IPython.display import Markdown, display
from huggingface_hub import hf_hub_download, snapshot_download
hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                        allow_patterns=[f'{n}/runs/*' for n in NAMES] + [f'{n}/v1_boundaries.json' for n in NAMES])
CTRL = json.load(open(hf_hub_download(REPO, 'nano-pollux-dual-v1.1/v1_boundaries.json', token=HF_TOKEN), encoding='utf-8'))
for n in NAMES:
    p = f'{hub}/{n}/v1_boundaries.json'
    rows = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
    display(Markdown(f'### {n}: {len(rows)} boundaries'))
    display(Markdown(V1.v12_table(rows, CTRL)))
    display(Markdown(V1.v12_marks_table(V1.v12_marks(rows, CTRL))))
%load_ext tensorboard
%tensorboard --logdir {hub}


### Changing the run

Edit the decision block in C1 (the dose: `NAMES = ['nano-pollux-dual-v1.2-p25']` or `-p75`; the session's hours; the shapes), rerun C1 (the install is cached), then C2 and C3. A craft already in the repo resumes from its last checkpoint; it never restarts. The control is the plain v1.1 craft's record on the Hub; a fresh control beside this run is a second notebook on `colab/nano_v1_1.ipynb` with `NAMES = ['nano-pollux-dual-v1.1']` under another name, if ever wanted.
